<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Efectos sobre el empleo, la pobreza y el recaudo tributario

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.
- `openpyxl`: para leer los valores calculados del libro de resultados.

In [1]:
import pandas as pd
import numpy as np
import openpyxl

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, tables

### Parámetros

Define el escenario de tasa de éxito exportador y la agrupación temporal de los resultados.

El escenario es el del quince por ciento, que es el libro que se conserva más reciente. La
agrupación es por quinquenios, que es la que usa el capítulo.

**Nota de auditoría:** el libro de resultados se construyó en Excel, no en la cadena, y la
versión con la que se produjeron el deck y el capítulo no se conserva. Las cifras que sale de
aquí no coinciden con las publicadas. Ver `tools/markdown/AUDITORIA_MIGRACION.md`, hallazgo 2.9.

In [3]:
version = 'V07'
escenario = '15'

# Los periodos NO son los que rotulan los entregables. El deck los llama periodos
# presidenciales (2025-28, 2029-32, ...) y el capitulo los llama quinquenios
# (2025-2029, 2030-2034, ...). Ninguno de los dos reproduce las cifras publicadas;
# esta particion si, exactamente. Ver tools/markdown/AUDITORIA_MIGRACION.md, hallazgo 2.10
periodos = [(2025, 2029), (2030, 2033), (2034, 2037), (2038, 2041), (2042, 2045)]

# cifras que reportan el deck y el capítulo, para contrastar
publicado_desempleo = [549_710, 823_419, 708_188, 295_304, 0]

### Lectura del libro de resultados

Carga las referencias de población y producto de 2024, y la serie anual de las variables macro.

La hoja aplica elasticidades constantes, castigadas en el tiempo, sobre la columna de producto
interno bruto adicional, que a su vez proviene del bootstrap de la matriz insumo-producto.

**Pasos realizados:**

1. **Referencias:** extrae de la hoja de números la población ocupada, la población en pobreza y el producto interno bruto de 2024, que son los denominadores de los porcentajes reportados.

2. **Serie anual:** lee por posición las columnas de la hoja de variables macro, que no tiene encabezados utilizables.

3. **Signo:** las tres series de personas vienen en negativo porque miden reducciones; se toman en valor absoluto.

In [4]:
archivo = raw / 'impacto_excel' / f'Nearshoring Resultados Númericos - {version} - {escenario}%.xlsx'
libro = openpyxl.load_workbook(archivo, data_only=True)

# Hoja de referencias: poblacion ocupada, poblacion en pobreza y PIB de 2024
referencias = {
    fila[0]: fila[1]
    for fila in libro['Números'].iter_rows(min_row=1, max_row=45, values_only=True)
    if isinstance(fila[0], str) and isinstance(fila[1], (int, float))
}

ocupados_2024 = referencias['Personas ocupadas en 2024']
pobres_2024 = referencias['Personas en Pobreza 2024']
pib_2024 = referencias['PIB 2024 a precios de 2015']

# Hoja de variables macro: una fila por año, columnas por posición
posiciones = {
    'anio': 0,
    'pib_adicional_pct': 3,
    'personas_desempleo': 7,
    'personas_pobreza': 11,
    'personas_pobreza_extrema': 14,
    'recaudo_total': 22,
    'recaudo_valor_presente': 23,
}

df_macro = pd.DataFrame([
    {nombre: fila[col] for nombre, col in posiciones.items()}
    for fila in libro['3 - Otras Variables Macro'].iter_rows(min_row=3, max_row=30, values_only=True)
    if isinstance(fila[0], (int, float)) and 2024 < fila[0] < 2050
])
df_macro['anio'] = df_macro['anio'].astype(int)

# las tres series de personas vienen en negativo porque miden reducciones
for columna in ['personas_desempleo', 'personas_pobreza', 'personas_pobreza_extrema']:
    df_macro[columna] = df_macro[columna].abs()

print(f"{version} al {escenario}%: {len(df_macro)} años, {df_macro['anio'].min()} a {df_macro['anio'].max()}")

V07 al 15%: 21 años, 2025 a 2045


### Tabla de efectos por quinquenio

Agrega la serie anual por quinquenio y expresa cada resultado como proporción de su referencia
de 2024. Corresponde a la tabla 2 del capítulo.

In [5]:
def periodo(anio):
    for inicio, fin in periodos:
        if inicio <= anio <= fin:
            return f"{inicio}-{fin}"
    return None


df_macro['periodo'] = df_macro['anio'].map(periodo)

df_tabla2 = (
    df_macro
    .groupby('periodo', sort=True)
    [['personas_desempleo', 'personas_pobreza', 'personas_pobreza_extrema', 'recaudo_total']]
    .sum()
    .reset_index()
)

df_tabla2['desempleo_pct'] = df_tabla2['personas_desempleo'] / ocupados_2024
df_tabla2['pobreza_pct'] = df_tabla2['personas_pobreza'] / pobres_2024
df_tabla2['pobreza_extrema_pct'] = df_tabla2['personas_pobreza_extrema'] / pobres_2024
df_tabla2['recaudo_pct_pib'] = df_tabla2['recaudo_total'] / pib_2024

df_tabla2

,periodo,personas_desempleo,personas_pobreza,personas_pobreza_extrema,recaudo_total,desempleo_pct,pobreza_pct,pobreza_extrema_pct,recaudo_pct_pib
0,2025-2029,"549,710.21","1,134,720.56","567,360.28","20,643,925,757,554.44",0.02,0.07,0.03,0.02
1,2030-2033,"823,419.46","1,272,853.96","636,426.98","28,770,155,328,885.01",0.04,0.08,0.04,0.03
2,2034-2037,"708,188.00","894,554.78","447,277.39","32,967,831,996,334.59",0.03,0.06,0.03,0.03
3,2038-2041,"295,303.52","122,518.17","61,259.08","32,921,764,404,062.54",0.01,0.01,0.00,0.03
4,2042-2045,0.00,0.00,0.00,"24,945,014,560,799.63",0.00,0.00,0.00,0.02


### Tabla de efectos acumulados

Suma los veintiún años para obtener el efecto total del período. Corresponde a la tabla 1 del
capítulo.

In [6]:
df_tabla1 = pd.DataFrame([
    {'Indicador': 'Reducción del desempleo',
     'Resultado acumulado': df_macro['personas_desempleo'].sum(),
     'Referencia': df_macro['personas_desempleo'].sum() / ocupados_2024},
    {'Indicador': 'Reducción de la pobreza monetaria',
     'Resultado acumulado': df_macro['personas_pobreza'].sum(),
     'Referencia': df_macro['personas_pobreza'].sum() / pobres_2024},
    {'Indicador': 'Reducción de la pobreza extrema',
     'Resultado acumulado': df_macro['personas_pobreza_extrema'].sum(),
     'Referencia': df_macro['personas_pobreza_extrema'].sum() / pobres_2024},
    {'Indicador': 'Aumento del recaudo tributario',
     'Resultado acumulado': df_macro['recaudo_valor_presente'].sum(),
     'Referencia': df_macro['recaudo_valor_presente'].sum() / pib_2024},
])

df_tabla1

,Indicador,Resultado acumulado,Referencia
0,Reducción del desempleo,"2,376,621.18",0.10
1,Reducción de la pobreza monetaria,"3,424,647.46",0.21
2,Reducción de la pobreza extrema,"1,712,323.73",0.11
3,Aumento del recaudo tributario,"102,958,942,168,941.20",0.10


### Contraste con lo publicado

Compara la reducción del desempleo por quinquenio con la que reportan el deck y el capítulo. La
diferencia mide cuánto se alejó el libro conservado de la versión con la que se escribieron los
entregables.

In [7]:
df_contraste = df_tabla2[['periodo', 'personas_desempleo']].copy()
df_contraste['publicado'] = publicado_desempleo
df_contraste['diferencia'] = df_contraste['personas_desempleo'] - df_contraste['publicado']

print(df_contraste.to_string(index=False))
print()
print(f"total regenerado: {df_contraste['personas_desempleo'].sum():,.0f} personas")
print(f"total publicado : {df_contraste['publicado'].sum():,.0f} personas")

  periodo  personas_desempleo  publicado  diferencia
2025-2029          549,710.21     549710        0.21
2030-2033          823,419.46     823419        0.46
2034-2037          708,188.00     708188       -0.00
2038-2041          295,303.52     295304       -0.48
2042-2045                0.00          0        0.00

total regenerado: 2,376,621 personas
total publicado : 2,376,621 personas


### Exportación

Guarda las dos tablas en LaTeX y en un libro de Excel con una hoja por tabla.

In [8]:
tables.mkdir(parents=True, exist_ok=True)

df_tabla1.to_latex(tables / 'macro_efectos_acumulados.tex', index=False, float_format="{:,.3f}".format)
df_tabla2.to_latex(tables / 'macro_efectos_por_periodo.tex', index=False, float_format="{:,.3f}".format)

with pd.ExcelWriter(tables / 'macro_empleo_pobreza_recaudo.xlsx') as writer:
    df_tabla1.to_excel(writer, sheet_name='Acumulado', index=False)
    df_tabla2.to_excel(writer, sheet_name='Por periodo', index=False)
    df_macro.to_excel(writer, sheet_name='Serie anual', index=False)